In [ ]:
from ollama import ChatResponse, chat, list as list_ollama_models
from pprint import pprint

In [ ]:
ollama_models = list_ollama_models()

In [ ]:
for _, model in enumerate(ollama_models.models):
    pprint(dict(model))

In [ ]:
# Swap this for the model you chose in Step 1 (the one your machine can hold)
MODEL = "qwen3:4b"

response: ChatResponse = chat(
    model=MODEL,
    messages=[
        {
            'role': 'user',
            'content': 'What is the difference between post training quantisation and quantisation aware training?',
        },
    ],
)

print(response.message.content)


## Activity A: structured output (messy text in, valid JSON out)

This is the "boring but real" product pattern: take text no script could parse and get back data your code can use.

First, try it YOUR way: write a prompt below that asks the model to turn `messy_text` into JSON. Run it and look closely at what comes back. Is the JSON valid? Did the model invent its own field names? Would a second run give the same shape?

In [ ]:
messy_text = """hey!! so the meetup got moved AGAIN lol. it's now thurs 7pm at
the Signal Cafe (the one on Mill Road not the station one). sarah + the two devs
from cambs AI are coming, maybe dan if his train isn't cancelled. bring adapters!!"""

# Your attempt: write your own prompt asking for JSON. Note think=False, see below.
your_prompt = "..."

response = chat(model=MODEL, think=False,
                messages=[{'role': 'user', 'content': your_prompt + "\n\n" + messy_text}])
print(response.message.content)


### Now constrain the shape

Prompt-only JSON drifts: different runs invent different schemas. Ollama can force the output to match a JSON schema instead. Two things matter with a thinking model like qwen3:

- `think=False` is required here. Schema-constrained output through the OpenAI-style `/v1` endpoint spends the whole token budget thinking and returns EMPTY content; the native client with `think=False` answers in seconds.
- The schema guarantees the SHAPE. Read the values and check: are they true?

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "event": {"type": "string"},
        "day": {"type": "string"},
        "time": {"type": "string"},
        "venue": {"type": "string"},
        "attendees": {"type": "array", "items": {"type": "string"}},
        "uncertain_attendees": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["event", "day", "time", "venue", "attendees"],
}

response = chat(model=MODEL, think=False, format=schema,
                messages=[{'role': 'user',
                           'content': 'Extract the event details from this message:\n' + messy_text}])
print(response.message.content)


In [ ]:
# The artifact: parsed data your code can use. This is the seam every
# closed "AI inbox" product is built on, running on your laptop.
import json as _json

event = _json.loads(response.message.content)
for person in event["attendees"]:
    print(f"invite: {person}")
print(f'where/when: {event["venue"]}, {event["day"]} {event["time"]}')

# Comprehension check: the schema made this well-formed. Is it also RIGHT?
# (Look for people listed as attendees who are only "maybe" coming.)
